# BTTS Model Training Notebook

This notebook trains the first practical BTTS model for the betPawa virtual English League data you are collecting.

Core rule used here:

```text
If MD 23 is live and we are predicting MD 24:
features must use results only up to MD 22 + MD 24 odds.
```

So during training we create every historical row the same way: `feature_cutoff_matchday = target_matchday - 2`.

## 1. Setup

Run this notebook from the project root, or from the `notebooks` folder. It expects your current files:

- `Match day results/match_results.csv`
- `odds/final_odds/final_odds.csv`

Run the dependency setup cell once if the notebook kernel says a package like `numpy` is missing.


In [ ]:
# Run this cell once if imports fail in this notebook kernel.
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "joblib": "joblib",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages into this notebook kernel:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Notebook dependencies are already installed.")


In [ ]:
from pathlib import Path
import json
import os
import warnings

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from joblib import dump, load
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss, classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_PATH = PROJECT_ROOT / "Match day results" / "match_results.csv"
ODDS_PATH = PROJECT_ROOT / "odds" / "final_odds" / "final_odds.csv"
RAW_ODDS_DIR = PROJECT_ROOT / "odds" / "all_odds_files"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

LIVE_GAP_MATCHDAYS = 2
PICK_THRESHOLD = 0.65
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Results file:", RESULTS_PATH)
print("Odds file:", ODDS_PATH)
print("Raw odds archive:", RAW_ODDS_DIR)

## 2. Load Results And Create Target

`target_btts` is the answer during training:

```text
1 = both teams scored
0 = one or both teams failed to score
```

In [ ]:
def load_results(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df["season_id"] = df["season_id"].astype(str)
    df["home_team"] = df["home_team"].astype(str).str.upper().str.strip()
    df["away_team"] = df["away_team"].astype(str).str.upper().str.strip()
    numeric_cols = [
        "matchday",
        "halftime_home_goals",
        "halftime_away_goals",
        "fulltime_home_goals",
        "fulltime_away_goals",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["matchday", "fulltime_home_goals", "fulltime_away_goals"])
    df["matchday"] = df["matchday"].astype(int)
    df["fulltime_home_goals"] = df["fulltime_home_goals"].astype(int)
    df["fulltime_away_goals"] = df["fulltime_away_goals"].astype(int)
    df = df.drop_duplicates(["season_id", "matchday", "home_team", "away_team"])
    df = df.sort_values(["season_id", "matchday", "home_team", "away_team"]).reset_index(drop=True)
    df["target_btts"] = ((df["fulltime_home_goals"] > 0) & (df["fulltime_away_goals"] > 0)).astype(int)
    return df


results = load_results(RESULTS_PATH)
display(results.head())
print("Rows:", len(results))
print("Seasons:", results["season_id"].nunique())
print("BTTS rate:", round(results["target_btts"].mean(), 3))

## 3. Load Odds

The notebook uses the source-of-truth final odds file:

```text
odds/final_odds/final_odds.csv
```

That file is already wide, with one row per unique fixture based on `home_team + away_team`.

Main odds columns used:

```text
odd_1, odd_x, odd_2, btts_yes, btts_no
```

Raw snapshots and capture history stay in `odds/all_odds_files/`, but they are not used here unless timing analysis is needed.


In [ ]:
ODDS_RAW_COLUMNS = ["odds_1", "odds_x", "odds_2", "odds_btts_yes", "odds_btts_no"]
ODDS_DERIVED_COLUMNS = [
    "implied_home_prob",
    "implied_draw_prob",
    "implied_away_prob",
    "implied_btts_yes_prob",
    "implied_btts_no_prob",
    "btts_market_margin",
    "market_btts_yes_prob_norm",
    "market_btts_no_prob_norm",
]
ODDS_FEATURE_COLUMNS = ODDS_RAW_COLUMNS + ODDS_DERIVED_COLUMNS


def add_implied_odds_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ODDS_RAW_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["implied_home_prob"] = np.where(df["odds_1"] > 0, 1 / df["odds_1"], np.nan)
    df["implied_draw_prob"] = np.where(df["odds_x"] > 0, 1 / df["odds_x"], np.nan)
    df["implied_away_prob"] = np.where(df["odds_2"] > 0, 1 / df["odds_2"], np.nan)
    df["implied_btts_yes_prob"] = np.where(df["odds_btts_yes"] > 0, 1 / df["odds_btts_yes"], np.nan)
    df["implied_btts_no_prob"] = np.where(df["odds_btts_no"] > 0, 1 / df["odds_btts_no"], np.nan)

    btts_total = df["implied_btts_yes_prob"] + df["implied_btts_no_prob"]
    df["btts_market_margin"] = btts_total - 1
    df["market_btts_yes_prob_norm"] = df["implied_btts_yes_prob"] / btts_total
    df["market_btts_no_prob_norm"] = df["implied_btts_no_prob"] / btts_total
    return df


def load_odds(path: Path) -> pd.DataFrame:
    if not path.exists():
        print("No final odds file found. Training will continue with missing odds features.")
        return pd.DataFrame(columns=["matchday", "home_team", "away_team"] + ODDS_FEATURE_COLUMNS)

    raw = pd.read_csv(path)
    raw.columns = [c.strip() for c in raw.columns]
    raw["home_team"] = raw["home_team"].astype(str).str.upper().str.strip()
    raw["away_team"] = raw["away_team"].astype(str).str.upper().str.strip()
    raw["matchday"] = pd.to_numeric(raw["matchday"], errors="coerce")
    raw = raw.dropna(subset=["matchday", "home_team", "away_team"])
    raw["matchday"] = raw["matchday"].astype(int)

    wide = raw.rename(
        columns={
            "odd_1": "odds_1",
            "odd_x": "odds_x",
            "odd_2": "odds_2",
            "btts_yes": "odds_btts_yes",
            "btts_no": "odds_btts_no",
        }
    )
    if "captured_at" in wide.columns:
        wide = wide.sort_values("captured_at")
    wide = wide.drop_duplicates(["home_team", "away_team"], keep="last")
    return add_implied_odds_features(wide)


odds = load_odds(ODDS_PATH)
display(odds.head())
print("Odds fixture snapshots:", len(odds))
print("Odds matchdays:", sorted(odds["matchday"].dropna().unique().tolist()) if len(odds) else [])


## 4. Feature Builder

This is the most important part of the notebook.

For each training match:

```text
target_matchday = the match we want to predict
feature_cutoff_matchday = target_matchday - 2
```

So if a historical row is MD 24, its features are generated from MD 1-22 only, then the actual MD 24 BTTS result becomes the target.

In [ ]:
TEAM_STATS_COLUMNS = [
    "rank",
    "played",
    "points",
    "wins",
    "draws",
    "losses",
    "goals_for",
    "goals_against",
    "goal_difference",
    "btts_rate_so_far",
    "scored_rate_so_far",
    "conceded_rate_so_far",
    "avg_goals_for_so_far",
    "avg_goals_against_so_far",
]

FORM_COLUMNS = [
    "form_points_last_5",
    "form_wins_last_5",
    "form_draws_last_5",
    "form_losses_last_5",
]

SEASON_FEATURE_COLUMNS = [
    "season_matches_so_far",
    "season_btts_rate_so_far",
    "season_avg_goals_per_match_so_far",
    "season_home_win_rate_so_far",
    "season_draw_rate_so_far",
    "season_away_win_rate_so_far",
]


def make_team_match_rows(history: pd.DataFrame) -> pd.DataFrame:
    home = history[["matchday", "home_team", "away_team", "fulltime_home_goals", "fulltime_away_goals", "target_btts"]].copy()
    home.columns = ["matchday", "team", "opponent", "goals_for", "goals_against", "btts"]
    home["side"] = "home"

    away = history[["matchday", "away_team", "home_team", "fulltime_away_goals", "fulltime_home_goals", "target_btts"]].copy()
    away.columns = ["matchday", "team", "opponent", "goals_for", "goals_against", "btts"]
    away["side"] = "away"

    team_rows = pd.concat([home, away], ignore_index=True)
    team_rows["points"] = np.select(
        [team_rows["goals_for"] > team_rows["goals_against"], team_rows["goals_for"] == team_rows["goals_against"]],
        [3, 1],
        default=0,
    )
    team_rows["win"] = (team_rows["points"] == 3).astype(int)
    team_rows["draw"] = (team_rows["points"] == 1).astype(int)
    team_rows["loss"] = (team_rows["points"] == 0).astype(int)
    team_rows["scored"] = (team_rows["goals_for"] > 0).astype(int)
    team_rows["conceded"] = (team_rows["goals_against"] > 0).astype(int)
    return team_rows


def build_team_stats(history: pd.DataFrame) -> pd.DataFrame:
    team_rows = make_team_match_rows(history)
    if team_rows.empty:
        return pd.DataFrame(columns=["team"] + TEAM_STATS_COLUMNS).set_index("team")

    stats = team_rows.groupby("team").agg(
        played=("team", "size"),
        points=("points", "sum"),
        wins=("win", "sum"),
        draws=("draw", "sum"),
        losses=("loss", "sum"),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
        btts_rate_so_far=("btts", "mean"),
        scored_rate_so_far=("scored", "mean"),
        conceded_rate_so_far=("conceded", "mean"),
        avg_goals_for_so_far=("goals_for", "mean"),
        avg_goals_against_so_far=("goals_against", "mean"),
    ).reset_index()
    stats["goal_difference"] = stats["goals_for"] - stats["goals_against"]
    stats = stats.sort_values(["points", "goal_difference", "goals_for", "team"], ascending=[False, False, False, True])
    stats["rank"] = np.arange(1, len(stats) + 1)
    return stats.set_index("team")


def build_form_stats(history: pd.DataFrame) -> pd.DataFrame:
    team_rows = make_team_match_rows(history).sort_values(["team", "matchday"])
    if team_rows.empty:
        return pd.DataFrame(columns=["team"] + FORM_COLUMNS).set_index("team")

    last5 = team_rows.groupby("team", group_keys=False).tail(5)
    form = last5.groupby("team").agg(
        form_points_last_5=("points", "sum"),
        form_wins_last_5=("win", "sum"),
        form_draws_last_5=("draw", "sum"),
        form_losses_last_5=("loss", "sum"),
    )
    return form


def build_season_stats(history: pd.DataFrame) -> dict:
    if history.empty:
        return {col: np.nan for col in SEASON_FEATURE_COLUMNS}

    home_win = history["fulltime_home_goals"] > history["fulltime_away_goals"]
    draw = history["fulltime_home_goals"] == history["fulltime_away_goals"]
    away_win = history["fulltime_home_goals"] < history["fulltime_away_goals"]
    total_goals = history["fulltime_home_goals"] + history["fulltime_away_goals"]
    return {
        "season_matches_so_far": len(history),
        "season_btts_rate_so_far": history["target_btts"].mean(),
        "season_avg_goals_per_match_so_far": total_goals.mean(),
        "season_home_win_rate_so_far": home_win.mean(),
        "season_draw_rate_so_far": draw.mean(),
        "season_away_win_rate_so_far": away_win.mean(),
    }


def get_team_values(table: pd.DataFrame, team: str, columns: list[str], prefix: str) -> dict:
    if team in table.index:
        return {f"{prefix}_{col}": table.loc[team, col] for col in columns}
    return {f"{prefix}_{col}": np.nan for col in columns}


def make_stats_bundle(results_df: pd.DataFrame, season_id: str, cutoff_matchday: int) -> dict:
    history = results_df[(results_df["season_id"] == str(season_id)) & (results_df["matchday"] <= cutoff_matchday)].copy()
    return {
        "team_stats": build_team_stats(history),
        "form_stats": build_form_stats(history),
        "season_stats": build_season_stats(history),
    }


def add_comparison_features(record: dict) -> dict:
    record["points_diff"] = record["home_points"] - record["away_points"]
    record["rank_diff"] = record["away_rank"] - record["home_rank"]
    record["goal_difference_diff"] = record["home_goal_difference"] - record["away_goal_difference"]
    record["goals_for_diff"] = record["home_goals_for"] - record["away_goals_for"]
    record["goals_against_diff"] = record["home_goals_against"] - record["away_goals_against"]
    record["form_diff"] = record["home_form_points_last_5"] - record["away_form_points_last_5"]
    record["btts_rate_diff"] = record["home_btts_rate_so_far"] - record["away_btts_rate_so_far"]
    record["scored_rate_diff"] = record["home_scored_rate_so_far"] - record["away_scored_rate_so_far"]
    record["conceded_rate_diff"] = record["home_conceded_rate_so_far"] - record["away_conceded_rate_so_far"]
    return record


def build_base_feature_record(match: pd.Series, bundle: dict, include_target: bool) -> dict:
    record = {
        "season_id": str(match["season_id"]),
        "target_matchday": int(match["matchday"]),
        "feature_cutoff_matchday": int(match["matchday"]) - LIVE_GAP_MATCHDAYS,
        "home_team": match["home_team"],
        "away_team": match["away_team"],
    }
    record.update(bundle["season_stats"])
    record.update(get_team_values(bundle["team_stats"], match["home_team"], TEAM_STATS_COLUMNS, "home"))
    record.update(get_team_values(bundle["team_stats"], match["away_team"], TEAM_STATS_COLUMNS, "away"))
    record.update(get_team_values(bundle["form_stats"], match["home_team"], FORM_COLUMNS, "home"))
    record.update(get_team_values(bundle["form_stats"], match["away_team"], FORM_COLUMNS, "away"))
    record = add_comparison_features(record)
    if include_target:
        record["target_btts"] = int(match["target_btts"])
    return record


## 5. Build Odds Template

For model training, odds are added where available.

Important practical point: your current odds history is small and has no `season_id`, so many training rows will have missing odds. That is acceptable for the first notebook because the model pipeline imputes missing values. As you capture more odds, these columns become more useful.

In [ ]:
def make_odds_template(odds_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    if odds_df.empty:
        return pd.DataFrame(columns=["matchday", "home_team", "away_team"] + ODDS_FEATURE_COLUMNS), ["matchday", "home_team", "away_team"]

    keys = ["home_team", "away_team"]
    value_cols = [c for c in ODDS_FEATURE_COLUMNS if c in odds_df.columns]
    template = odds_df.groupby(keys, as_index=False)[value_cols].median(numeric_only=True)
    return template, keys


odds_template, odds_join_keys = make_odds_template(odds)
print("Odds join keys:", odds_join_keys)
display(odds_template.head())

## 6. Create Training Rows

One row means one fixture to predict.

Example training logic:

```text
Training row target: Season 138522, MD 24, ARS vs LIV
Feature data used: Season 138522, MD 1-22 only
Odds used: MD 24 ARS vs LIV odds if captured
Target: actual MD 24 BTTS result
```

In [ ]:
def build_training_table(results_df: pd.DataFrame, odds_template_df: pd.DataFrame, odds_keys: list[str]) -> pd.DataFrame:
    records = []
    stats_cache = {}

    for _, match in results_df.iterrows():
        target_md = int(match["matchday"])
        cutoff_md = target_md - LIVE_GAP_MATCHDAYS
        if cutoff_md < 1:
            continue

        cache_key = (str(match["season_id"]), cutoff_md)
        if cache_key not in stats_cache:
            stats_cache[cache_key] = make_stats_bundle(results_df, match["season_id"], cutoff_md)

        record = build_base_feature_record(match, stats_cache[cache_key], include_target=True)
        records.append(record)

    feature_df = pd.DataFrame(records)
    if feature_df.empty:
        raise ValueError("No training rows were built. Check matchday coverage and LIVE_GAP_MATCHDAYS.")

    join_left = odds_keys.copy()
    merged = feature_df.merge(odds_template_df, on=join_left, how="left", suffixes=("", "_odds"))

    for col in ODDS_FEATURE_COLUMNS:
        if col not in merged.columns:
            merged[col] = np.nan

    return merged


training_df = build_training_table(results, odds_template, odds_join_keys)
display(training_df.head())
print("Training rows:", len(training_df))
print("Rows with BTTS odds:", training_df[["odds_btts_yes", "odds_btts_no"]].notna().all(axis=1).sum())
print("Target BTTS rate:", round(training_df["target_btts"].mean(), 3))

## 7. Final Feature Columns

These are the columns actually fed into the model. Tracking columns and the target are not fed into the model.

In [ ]:
TRACKING_COLUMNS = ["season_id", "target_matchday", "feature_cutoff_matchday", "home_team", "away_team"]
TARGET_COLUMN = "target_btts"

candidate_features = [
    col
    for col in training_df.columns
    if col not in TRACKING_COLUMNS + [TARGET_COLUMN]
]
candidate_features = [col for col in candidate_features if pd.api.types.is_numeric_dtype(training_df[col])]

# If a feature has no values in training, the model cannot learn from it yet.
# This commonly happens with odds until final_odds.csv has matching fixtures.
ALL_MISSING_FEATURES = [col for col in candidate_features if training_df[col].notna().sum() == 0]
MODEL_FEATURES = [col for col in candidate_features if col not in ALL_MISSING_FEATURES]

feature_summary = pd.DataFrame(
    {
        "feature": candidate_features,
        "non_null_rows": [int(training_df[col].notna().sum()) for col in candidate_features],
        "used_by_model": [col in MODEL_FEATURES for col in candidate_features],
    }
)
display(feature_summary)
print("Number of model features:", len(MODEL_FEATURES))
print("Excluded all-missing features:", ALL_MISSING_FEATURES)

## 8. Train / Validate Split

We use the older rows for training and the latest rows for validation. This is closer to real prediction than a random split.

In [ ]:
ordered = training_df.sort_values(["season_id", "target_matchday", "home_team", "away_team"]).reset_index(drop=True)
split_index = int(len(ordered) * 0.8)

train_df = ordered.iloc[:split_index].copy()
valid_df = ordered.iloc[split_index:].copy()

X_train = train_df[MODEL_FEATURES]
y_train = train_df[TARGET_COLUMN]
X_valid = valid_df[MODEL_FEATURES]
y_valid = valid_df[TARGET_COLUMN]

print("Train rows:", len(train_df))
print("Validation rows:", len(valid_df))
print("Train BTTS rate:", round(y_train.mean(), 3))
print("Validation BTTS rate:", round(y_valid.mean(), 3))

## 9. Train Model

This section trains several candidate models on the same features and validation split, then keeps the one with the highest validation accuracy.

In [ ]:
candidate_models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGradientBoostingClassifier": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "ExtraTreesClassifier": ExtraTreesClassifier(
        n_estimators=400,
        min_samples_leaf=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "SoftVotingEnsemble": VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
            ("gb", GradientBoostingClassifier(random_state=RANDOM_STATE)),
            ("hgb", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
            ("rf", RandomForestClassifier(n_estimators=400, min_samples_leaf=8, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)),
            ("et", ExtraTreesClassifier(n_estimators=400, min_samples_leaf=8, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)),
        ],
        voting="soft",
    ),
}

model_results = []
trained_models = {}

for model_name, estimator in candidate_models.items():
    candidate = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", estimator),
        ]
    )
    candidate.fit(X_train, y_train)
    prob_yes = candidate.predict_proba(X_valid)[:, 1]
    pred = (prob_yes >= 0.5).astype(int)
    trained_models[model_name] = {"model": candidate, "prob": prob_yes, "pred": pred}
    model_results.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_valid, pred),
            "roc_auc": roc_auc_score(y_valid, prob_yes) if y_valid.nunique() == 2 else np.nan,
            "brier_score": brier_score_loss(y_valid, prob_yes),
        }
    )

model_comparison = pd.DataFrame(model_results).sort_values(
    ["accuracy", "brier_score"],
    ascending=[False, True],
).reset_index(drop=True)
display(model_comparison)

best_model_name = model_comparison.loc[0, "model"]
model = trained_models[best_model_name]["model"]
valid_prob = trained_models[best_model_name]["prob"]
valid_pred = trained_models[best_model_name]["pred"]

print("Best model:", best_model_name)
print("Accuracy:", round(accuracy_score(y_valid, valid_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_valid, valid_prob), 3) if y_valid.nunique() == 2 else "not available")
print("Brier score:", round(brier_score_loss(y_valid, valid_prob), 3))
print("Confusion matrix:")
print(confusion_matrix(y_valid, valid_pred))
print("\nClassification report:")
print(classification_report(y_valid, valid_pred, target_names=["BTTS No", "BTTS Yes"]))

## 10. Pick / Skip Threshold Check

This matters more than raw accuracy. Low failure comes from skipping weak matches.

`confidence = max(prob_yes, prob_no)`

In [ ]:
def evaluate_thresholds(y_true: pd.Series, prob_yes: np.ndarray) -> pd.DataFrame:
    rows = []
    for threshold in np.arange(0.50, 0.81, 0.05):
        pred = (prob_yes >= 0.5).astype(int)
        confidence = np.maximum(prob_yes, 1 - prob_yes)
        picked = confidence >= threshold
        if picked.sum() == 0:
            hit_rate = np.nan
        else:
            hit_rate = (pred[picked] == y_true.to_numpy()[picked]).mean()
        rows.append(
            {
                "threshold": round(float(threshold), 2),
                "picked_matches": int(picked.sum()),
                "coverage": round(float(picked.mean()), 3),
                "picked_hit_rate": np.nan if pd.isna(hit_rate) else round(float(hit_rate), 3),
            }
        )
    return pd.DataFrame(rows)


threshold_report = evaluate_thresholds(y_valid, valid_prob)
display(threshold_report)

## 11. Save Model

The notebook saves two important files:

- `models/btts_model.joblib`
- `models/btts_feature_columns.json`

Streamlit will load these later.

In [ ]:
model_path = MODEL_DIR / "btts_model.joblib"
columns_path = MODEL_DIR / "btts_feature_columns.json"

dump(model, model_path)
columns_payload = {
    "target": TARGET_COLUMN,
    "feature_columns": MODEL_FEATURES,
    "excluded_all_missing_features": ALL_MISSING_FEATURES,
    "selected_model_name": best_model_name,
    "model_comparison": model_comparison.round(4).to_dict(orient="records"),
    "live_gap_matchdays": LIVE_GAP_MATCHDAYS,
    "pick_threshold": PICK_THRESHOLD,
    "notes": "For live MD 24 prediction while MD 23 is playing, build features using results up to MD 22 and MD 24 odds.",
}
columns_path.write_text(json.dumps(columns_payload, indent=2), encoding="utf-8")

print("Saved model:", model_path)
print("Saved feature columns:", columns_path)

## 12. Live Prediction Example

This is the part Streamlit will later copy.

When MD 23 is live and you are betting MD 24:

```text
LIVE_SEASON_ID = current season
TARGET_MATCHDAY = 24
FEATURE_CUTOFF_MATCHDAY = 22
```

The saved model is not retrained here. We only build the latest feature rows and call `predict_proba`.

In [ ]:
def latest_season_id(results_df: pd.DataFrame) -> str:
    numeric = pd.to_numeric(results_df["season_id"], errors="coerce")
    if numeric.notna().any():
        return str(int(numeric.max()))
    return str(results_df["season_id"].sort_values().iloc[-1])


def latest_fixture_odds(odds_df: pd.DataFrame, target_matchday: int | None = None) -> pd.DataFrame:
    if odds_df.empty:
        raise ValueError("No odds rows available. Capture upcoming odds first.")
    if target_matchday is None:
        target_matchday = int(odds_df["matchday"].max())
    fixtures = odds_df[odds_df["matchday"] == target_matchday].copy()
    if fixtures.empty:
        raise ValueError(f"No odds rows found for matchday {target_matchday}.")
    if "captured_at" in fixtures.columns:
        fixtures = fixtures.sort_values("captured_at")
    fixtures = fixtures.drop_duplicates(["matchday", "home_team", "away_team"], keep="last")
    return fixtures[["matchday", "home_team", "away_team"] + [c for c in ODDS_FEATURE_COLUMNS if c in fixtures.columns]].copy()


def build_live_prediction_table(
    results_df: pd.DataFrame,
    fixture_odds_df: pd.DataFrame,
    season_id: str,
    feature_cutoff_matchday: int,
) -> pd.DataFrame:
    bundle = make_stats_bundle(results_df, season_id, feature_cutoff_matchday)
    records = []
    for _, fixture in fixture_odds_df.iterrows():
        match = pd.Series(
            {
                "season_id": season_id,
                "matchday": int(fixture["matchday"]),
                "home_team": fixture["home_team"],
                "away_team": fixture["away_team"],
            }
        )
        record = build_base_feature_record(match, bundle, include_target=False)
        record["feature_cutoff_matchday"] = feature_cutoff_matchday
        for col in ODDS_FEATURE_COLUMNS:
            record[col] = fixture[col] if col in fixture.index else np.nan
        records.append(record)
    return pd.DataFrame(records)


# Edit these when you know the current situation.
LIVE_SEASON_ID = latest_season_id(results)
TARGET_MATCHDAY = None  # None means use the latest matchday available in final_odds.csv

fixture_odds = latest_fixture_odds(odds, TARGET_MATCHDAY)
TARGET_MATCHDAY = int(fixture_odds["matchday"].iloc[0])
FEATURE_CUTOFF_MATCHDAY = TARGET_MATCHDAY - LIVE_GAP_MATCHDAYS

live_df = build_live_prediction_table(results, fixture_odds, LIVE_SEASON_ID, FEATURE_CUTOFF_MATCHDAY)
display(live_df[["season_id", "target_matchday", "feature_cutoff_matchday", "home_team", "away_team", "odds_btts_yes", "odds_btts_no"]].head())
print("Live season:", LIVE_SEASON_ID)
print("Predicting matchday:", TARGET_MATCHDAY)
print("Using results up to matchday:", FEATURE_CUTOFF_MATCHDAY)

In [ ]:
saved_model = load(MODEL_DIR / "btts_model.joblib")
saved_columns = json.loads((MODEL_DIR / "btts_feature_columns.json").read_text(encoding="utf-8"))["feature_columns"]

for col in saved_columns:
    if col not in live_df.columns:
        live_df[col] = np.nan

live_prob_yes = saved_model.predict_proba(live_df[saved_columns])[:, 1]
predictions = live_df[["season_id", "target_matchday", "feature_cutoff_matchday", "home_team", "away_team", "odds_btts_yes", "odds_btts_no"]].copy()
predictions["btts_yes_probability"] = live_prob_yes
predictions["prediction"] = np.where(predictions["btts_yes_probability"] >= 0.5, "Yes", "No")
predictions["confidence"] = np.maximum(predictions["btts_yes_probability"], 1 - predictions["btts_yes_probability"])
predictions["decision"] = np.where(predictions["confidence"] >= PICK_THRESHOLD, "Pick", "Skip")

predictions = predictions.sort_values(["decision", "confidence"], ascending=[True, False]).reset_index(drop=True)
display(predictions)

prediction_path = OUTPUT_DIR / f"btts_predictions_md{TARGET_MATCHDAY}.csv"
predictions.to_csv(prediction_path, index=False)
print("Saved predictions:", prediction_path)

## What Streamlit Will Consume Later

Streamlit will not train the model. It will only do this:

```text
1. Load models/btts_model.joblib
2. Load models/btts_feature_columns.json
3. Read latest results CSV and latest odds CSV
4. Build live features using target_matchday - 2
5. Show BTTS probability, prediction, confidence, Pick/Skip
```

That is how the notebook model is consumed.